# Practical 3: CNN image classification on MNIST

**What we are doing.** The task is the same as P2: read a handwritten digit. The difference is *how the network looks at the picture*. P2 flattened it immediately and lost which pixel was next to which. A **CNN** slides small **3×3 filters** over the image (each one a pattern detector, such as an edge or a curve), shrinks the result with **pooling**, and only then flattens and classifies. Accuracy is about **99%**, against about 97% for P2.

```
28x28x1 → Conv 32 (3x3) → 26x26x32 → MaxPool → 13x13x32
        → Conv 64 (3x3) → 11x11x64 → MaxPool → 5x5x64
        → Flatten 1600 → Dense 128 → Dropout 0.5 → Dense 10 softmax
```
**Skeleton:** imports → load → preprocess → **model** → compile → fit → evaluate → plot/predict

> **Run locally in Jupyter?** Install once: `%pip install tensorflow numpy pandas matplotlib scikit-learn`, then *Kernel → Restart Kernel*.
> **In Colab:** *Runtime → Change runtime type → T4 GPU*. Everything is preinstalled.

## Imports, load, preprocess
**New:** `reshape(-1, 28, 28, 1)`. Conv layers expect (height, width, **channels**): 1 = grayscale, 3 = RGB. `-1` means "however many images there are".

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# a. Load and preprocess
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1) / 255.0   # add a channel axis: Conv2D wants (h, w, channels)
x_test = x_test.reshape(-1, 28, 28, 1) / 255.0
print(x_train.shape, x_test.shape)

## b. Model (the main change)
- **Conv2D(32, (3,3))**: 32 learnable 3×3 filters, each producing a *feature map* of where its pattern appears.
- **MaxPooling2D((2,2))**: keeps the biggest value in each 2×2 square, which halves the size. 0 params.
- **Dropout(0.5)**: during training only, randomly switches off half the neurons, which reduces overfitting.
- Memory trick: *Conv–Pool, Conv–Pool, Flatten, Dense, Drop, Dense*, with **32 → 64 → 128 → 10**.

| Layer | Params | How |
|---|---|---|
| Conv 32 | 320 | 32 × (3·3·1 + 1) |
| Conv 64 | 18,496 | 64 × (3·3·32 + 1) |
| Dense 128 | 204,928 | 1600 × 128 + 128 |
| Dense 10 | 1,290 | 128 × 10 + 10 |

In [ ]:
# b. Model architecture
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),   # 32 filters of 3x3 -> 26x26x32
    layers.MaxPooling2D((2, 2)),                    # halve -> 13x13x32
    layers.Conv2D(64, (3, 3), activation='relu'),   # -> 11x11x64
    layers.MaxPooling2D((2, 2)),                    # -> 5x5x64
    layers.Flatten(),                               # -> 1600
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),                            # randomly switch off half, only while training
    layers.Dense(10, activation='softmax'),
])
model.summary()

## c. Compile and train
**Adam** adapts the step size for each weight and usually learns faster. P3's statement doesn't require SGD. 54,000 ÷ 64 = 844 steps per epoch.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# c. Train
history = model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.1)

## d. Estimate performance and show predictions
Expect about **99%**. In the picture titles, P = predicted and A = actual.

In [ ]:
# d. Estimate performance
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test loss:", test_loss, " Test accuracy:", test_acc)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend()
plt.show()

# Show 10 test images with predicted (P) and actual (A) labels
pred = model.predict(x_test[:10]).argmax(axis=1)
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.title(f"P{pred[i]} A{y_test[i]}")
    plt.axis('off')
plt.show()

---
## Understand it: poke cells
**A. What the first conv layer sees.** Each of the 32 filters lights up where it finds its pattern. 28 → 26 because a 3×3 filter can't be centred on the border pixels.

In [ ]:
import numpy as np
fmap = keras.Model(model.inputs, model.layers[0].output)(x_test[:1]).numpy()
print(fmap.shape)       # (1, 26, 26, 32)
plt.figure(figsize=(12, 3))
for i in range(8):
    plt.subplot(1, 8, i + 1); plt.imshow(fmap[0, :, :, i], cmap='gray'); plt.axis('off')
plt.show()

**B. Max pooling on a 4×4 example:** keeps the biggest value of each 2×2 window, giving `[[4, 2], [2, 7]]`.

In [ ]:
x = np.array([[1, 3, 2, 0], [4, 2, 1, 1], [0, 1, 5, 6], [2, 2, 7, 1]], dtype='float32').reshape(1, 4, 4, 1)
print(layers.MaxPooling2D((2, 2))(x).numpy().reshape(2, 2))

---
## Break it: remove pooling
Without the two MaxPooling2D layers, Flatten becomes 24·24·64 = **36,864** and Dense(128) jumps to about **4.7 million** params. Training is slower and more likely to overfit. That's why pooling exists.

In [ ]:
no_pool = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
])
no_pool.summary()

---
## Viva answers
**Q1. What does a convolution layer do? Params of Conv2D(32, 3×3) on grayscale?** It slides 32 learnable 3×3 filters over the image, taking a dot product at each position, to make 32 feature maps. 32 × (9 + 1) = **320**.

**Q2. Why pooling, and why dropout?** Pooling down-samples (max of each 2×2), which cuts size and computation and gives a little position tolerance. Dropout randomly switches off 50% of the neurons during training only, which reduces overfitting.

**Q3. Why does a CNN beat a feedforward network on images?** The FFNN flattens first and loses the 2-D neighbourhood. A CNN keeps it, and each filter is shared across the whole image, so it needs fewer parameters and finds a pattern wherever it is.

> **Classmate's code** uses `to_categorical` + `categorical_crossentropy`, which is equivalent to our integer labels + `sparse_categorical_crossentropy`.